In [2]:
# Notebook-friendly miner (auto-runs)
from __future__ import annotations

import csv, json, os, random, re, subprocess, sys, time, shutil
from concurrent.futures import ThreadPoolExecutor as PoolExecutor, as_completed
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

# ---------------------------
# Config (EDIT THESE PATHS)
# ---------------------------
WORK_ROOT = Path(r"C:\Temp_Instru")
INPUT_CLONES = WORK_ROOT / "clonesV9.0"
OUTPUT_MINE = WORK_ROOT / "mineV9.0"

# Cutoff (UTC recommended). Used for git --before and for censoring at the last day.
CUTOFF_ISO = "2025-08-10 23:59:59 +0000"

# Performance / behavior toggles
MAX_REPOS = 0                      # 0 = all
MAX_WORKERS = 6                    # keep modest to avoid disk/AV thrash
RESUME_IF_EXISTS = True            # skip repos already mined
SUPPRESS_EMPTY_ROWS = True         # do not emit NONE-state rows in timeline
CAPTURE_EMPTY_GAPS = True          # capture NONE episodes as empty_gaps
EMIT_NONE_STATE = False            # if True, emit explicit NONE rows (usually False)

# Blob size guards (YAML/Gradle/scripts)
BLOB_MAX_SIZE = 2_000_000          # soft skip for large blobs (except high-priority CI)
HARD_MAX_SIZE = 5_000_000          # absolute cap (safety)

# CI YAML dirs/files we never skip (even if large)
HIGH_PRIORITY_CI_PATHS = {
    ".github/workflows", ".gitlab-ci.yml", "azure-pipelines.yml",
    ".circleci/config.yml", ".bitrise.yml", ".travis.yml",
}

# ---------------------------
# Quick startup checks (fail fast with clear messages)
# ---------------------------
def _failfast_checks():
    if shutil.which("git") is None:
        print("[fatal] Git not found in PATH. Install Git and/or add it to PATH.", file=sys.stderr, flush=True)
        raise SystemExit(1)
    if not WORK_ROOT.exists():
        print(f"[fatal] WORK_ROOT does not exist: {WORK_ROOT}", file=sys.stderr, flush=True)
        raise SystemExit(1)
    if not INPUT_CLONES.exists():
        print(f"[warn] INPUT_CLONES does not exist yet: {INPUT_CLONES}", flush=True)
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)

_failfast_checks()
print("[ok] Pre-flight checks passed.", flush=True)

# ---------------------------
# File classifiers
# ---------------------------
YAML_EXTS = (".yml", ".yaml")
GRADLE_NAMES = {"build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts"}
GRADLE_EXTS = (".gradle", ".gradle.kts")
SCRIPT_EXTS = (".sh", ".bat", ".cmd", ".ps1")
CI_BUILD_SPECIAL = {"Jenkinsfile", ".travis.yml", "azure-pipelines.yml", ".gitlab-ci.yml", "circle.yml"}
XML_BUILD_FILES = {"pom.xml", "build.xml", "config.xml"}

def is_yaml(path: str) -> bool:
    return os.path.splitext(path)[1].lower() in YAML_EXTS or os.path.basename(path) in CI_BUILD_SPECIAL

def is_gradle(path: str) -> bool:
    name = os.path.basename(path)
    ext = os.path.splitext(path)[1].lower()
    return name in GRADLE_NAMES or ext in GRADLE_EXTS

def is_script(path: str) -> bool:
    ext = os.path.splitext(path)[1].lower()
    return ext in SCRIPT_EXTS or os.path.basename(path) in ("Jenkinsfile",)

def is_ci_xml_or_build_xml(path: str) -> bool:
    return os.path.basename(path).lower() in {n.lower() for n in XML_BUILD_FILES}

def is_relevant_file(path: str) -> bool:
    return is_yaml(path) or is_gradle(path) or is_script(path) or is_ci_xml_or_build_xml(path)

def is_high_priority_ci_path(path: str) -> bool:
    norm = path.replace("\\", "/")
    base = os.path.basename(norm)
    if base in HIGH_PRIORITY_CI_PATHS:
        return True
    for root in HIGH_PRIORITY_CI_PATHS:
        if norm.startswith(root.rstrip("/") + "/"):
            return True
    return False

# ---------------------------
# Regex heuristics (states & proxies)
# ---------------------------
EMU_COMMUNITY = [
    re.compile(r"reactivecircus\s*/\s*android[-_]emulator[-_]runner", re.IGNORECASE),
    re.compile(r"malinskiy\s*/\s*action[-_]android", re.IGNORECASE),
    re.compile(r"malinskiy\s*/\s*android[-_]emulator[-_]runner", re.IGNORECASE),
    re.compile(r"\buses\s*:\s*malinskiy/action-android/emulator-run-cmd@[\w.\-]+", re.IGNORECASE),
]
EMU_CUSTOM = [
    re.compile(r"\bemulator(\.exe)?\s*-[A-Za-z]", re.IGNORECASE),
    re.compile(r"\bavdmanager\b", re.IGNORECASE),
    re.compile(r"\bsdkmanager\b", re.IGNORECASE),
    re.compile(r"\badb\b", re.IGNORECASE),
    re.compile(r"\bandroid\s+create\s+avd\b", re.IGNORECASE),
    re.compile(r"\bemulator-headless\b", re.IGNORECASE),
    re.compile(r"system-images;android-\d+;google_apis(?:;x86_64|;x86)?", re.IGNORECASE),
    re.compile(r"echo\s+no\s*\|\s*avdmanager\s+create\s+avd", re.IGNORECASE),
    re.compile(r"emulator\s+-list-avds", re.IGNORECASE),
    re.compile(r"\bandroid\s+update\s+sdk\b", re.IGNORECASE),
    re.compile(r"\bandroid\s+update\s+avd\b", re.IGNORECASE),
    re.compile(r"(?mi)^\s*\S*(?:sudo\s+)?(?:bash|sh|pwsh|powershell)?[^#\n;]*\badb\s+wait[- ]?for[- ]?device\b"),
    re.compile(r"(?mi)^\s*\S*(?:sudo\s+)?(?:bash|sh|pwsh|powershell)?[^#\n;]*\badb\s+-s\s+(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)\b"),
    re.compile(r"(?mi)^\s*\S*(?:sudo\s+)?(?:bash|sh|pwsh|powershell)?[^#\n;]*\bemulator\b[^\n]*(?:-avd\s+\S+|@\S+)"),
    re.compile(r"(?mi)^\s*(?:\./)?android-wait-for-emulator\b"),
    re.compile(r"(?mi)^\s*start-emulator\.sh\b"),
    re.compile(r"(?mi)^\s*circle-android\s+wait-for-boot\b"),
    re.compile(r"(?i)\bconnectedandroidtest\b"),
    re.compile(r"(?i)\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b"),
    re.compile(r"(?i)\bdevicecheck\b|\balldevicechecks\b"),
]
GMD_GRADLE = [
    re.compile(r"\btestOptions\s*\{[^}]*managedDevices\s*\{", re.IGNORECASE | re.DOTALL),
    re.compile(r"\bmanagedDevices\s*\{", re.IGNORECASE),
    re.compile(r"\bManagedVirtualDevice\b", re.IGNORECASE),
    re.compile(r"\bdeviceGroups?\b", re.IGNORECASE),
]
THIRDPARTY = [
    re.compile(r"\bgcloud\s+firebase\s+test\s+android\s+run\b", re.IGNORECASE),
    re.compile(r"\bbrowserstack\b", re.IGNORECASE),
    re.compile(r"\bsaucectl\b|\bsaucelabs\b", re.IGNORECASE),
    re.compile(r"\bdevice\s*farm\b", re.IGNORECASE),
]
YAML_VENDOR_EMU_CONTEXT = [
    re.compile(r"(?mi)\bimage\s*:\s*(?:reactivecircus|cirrusci|budtmo)/android[-\w:]*"),
    re.compile(r"(?mi)\btask\s*:\s*AndroidToolInstaller@", re.IGNORECASE),
]
RX_RUNS_ON = re.compile(r"(?mi)^\s*runs-on\s*:\s*(.+)$")
RX_API_LEVEL = re.compile(r"(?i)\bandroid[-_ ]?(\d{2})\b|system-images;android-(\d{2})\b")
RX_LIST_ITEM = re.compile(r"(?mi)^\s*-\s")
RX_RETRY = re.compile(r"(?i)\bretry\b|\bmax-attempts\b|\bflaky\b")
RX_TIMEOUT = re.compile(r"(?i)\btimeout[- ]?minutes\b|\btimeout\b")
RX_HEADLESS = re.compile(r"(?i)\bheadless\b|\b-no-window\b|\b-no-boot-anim\b")
RX_WAIT_FOR_DEVICE = re.compile(r"(?i)\badb\s+wait[- ]?for[- ]?device\b")

# ---------------------------
# Git helpers
# ---------------------------
def run_git(repo: Path, args: List[str], text: bool = True, check: bool = True, retries: int = 2) -> subprocess.CompletedProcess:
    last_exc: Optional[BaseException] = None
    for attempt in range(retries + 1):
        try:
            return subprocess.run(
                ["git", "-C", str(repo)] + args,
                stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=text, check=check,
                encoding="utf-8" if text else None,
                errors="replace" if text else None,
            )
        except subprocess.CalledProcessError as e:
            last_exc = e
            time.sleep(0.1 * (attempt + 1) + random.random() * 0.1)
        except Exception as e:
            last_exc = e
            time.sleep(0.05)
    if isinstance(last_exc, subprocess.CalledProcessError) and not check:
        return last_exc  # type: ignore[return-value]
    raise last_exc  # type: ignore[misc]

def list_repos(root: Path) -> List[Path]:
    out: List[Path] = []
    for p, dirs, _files in os.walk(root):
        pth = Path(p)
        if (pth / ".git").exists():
            out.append(pth)
            dirs[:] = []
    return out

def cutoff_heads(repo: Path, cutoff_iso: str) -> List[str]:
    refs_raw = run_git(repo, ["for-each-ref", "--format=%(refname:short)", "refs/remotes/origin"]).stdout
    refs = [r for r in refs_raw.splitlines() if r and r != "origin/HEAD" and not r.startswith("origin/pr/")]
    shas: List[str] = []
    for r in refs:
        cp = run_git(repo, ["rev-list", "-n1", "--first-parent", f"--before={cutoff_iso}", r], check=False)
        sha = (cp.stdout or "").strip()
        if sha:
            shas.append(sha)
    if not shas:
        cp = run_git(repo, ["rev-list", "-n1", "--before", cutoff_iso, "--all"], check=False)
        sha = (cp.stdout or "").strip()
        if sha:
            shas.append(sha)
    return shas

def earliest_commit(repo: Path) -> Optional[str]:
    try:
        s = run_git(repo, ["rev-list", "--max-parents=0", "--all"]).stdout.strip()
        return s.splitlines()[0] if s else None
    except Exception:
        return None

def get_commit_date_iso(repo: Path, commit: str) -> Optional[str]:
    try:
        return run_git(repo, ["show", "-s", "--format=%cI", commit]).stdout.strip() or None
    except Exception:
        return None

def _pathspecs_yaml() -> List[str]:
    return [":(glob)**/*.yml", ":(glob)**/*.yaml", ".travis.yml", "azure-pipelines.yml", ".gitlab-ci.yml", "circle.yml", ".bitrise.yml"]

def _pathspecs_gradle() -> List[str]:
    return ["build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts", ":(glob)**/*.gradle", ":(glob)**/*.gradle.kts"]

def _pathspecs_scripts_and_xml() -> List[str]:
    return ["Jenkinsfile", ":(glob)**/*.sh", ":(glob)**/*.bat", ":(glob)**/*.cmd", ":(glob)**/*.ps1", ":(glob)**/pom.xml", ":(glob)**/build.xml", ":(glob)**/config.xml"]

def _pathspecs_all_relevant() -> List[str]:
    return _pathspecs_yaml() + _pathspecs_gradle() + _pathspecs_scripts_and_xml()

# --- FIX FOR WINDOWS CMD-LINE LENGTH (WinError 206) ---
# Use --all with --before instead of enumerating every head on the CLI.
def commits_touching_relevant_from_heads(repo: Path, cutoff_iso: str) -> List[str]:
    pathspecs = _pathspecs_all_relevant()
    s = run_git(
        repo,
        ["rev-list", "--reverse", f"--before={cutoff_iso}", "--all", "--"] + pathspecs
    ).stdout
    return [c for c in s.splitlines() if c.strip()]

@lru_cache(maxsize=250_000)
def list_tree_entries_cached(repo_path: str, treeish: str) -> List[Tuple[str, str]]:
    repo = Path(repo_path)
    pathspecs = _pathspecs_all_relevant()
    try:
        raw = run_git(repo, ["ls-tree", "-r", "-z", treeish, "--"] + pathspecs).stdout
    except Exception:
        raw = ""
    out: List[Tuple[str, str]] = []
    for entry in raw.split("\x00"):
        if not entry or "\t" not in entry:
            continue
        meta, path = entry.split("\t", 1)
        parts = meta.split()
        if len(parts) < 3:
            continue
        sha = parts[2]
        out.append((sha, path))
    return out

@lru_cache(maxsize=250_000)
def read_blob_cached(repo_path: str, sha: str) -> Optional[str]:
    repo = Path(repo_path)
    try:
        return run_git(repo, ["cat-file", "-p", sha]).stdout
    except Exception:
        return None

# ---------------------------
# Feature extraction (proxies)
# ---------------------------
def extract_proxies_from_text(path: str, text: str) -> Dict:
    feat: Dict[str, object] = {
        "yaml_loc": 0,
        "gradle_loc": 0,
        "runs_on": set(),
        "matrix_width_hint": 0,
        "api_levels": set(),
        "reliability_flags": set(),
    }
    lines = text.splitlines()
    if is_yaml(path):
        feat["yaml_loc"] = len(lines)
        for m in RX_RUNS_ON.finditer(text):
            feat["runs_on"].add(m.group(1).strip())  # type: ignore[attr-defined]
        list_items = len(RX_LIST_ITEM.findall(text))
        feat["matrix_width_hint"] = max(feat["matrix_width_hint"], list_items)  # type: ignore[index]
    elif is_gradle(path):
        feat["gradle_loc"] = len(lines)
    for m in RX_API_LEVEL.finditer(text):
        for g in m.groups():
            if g and g.isdigit():
                try:
                    feat["api_levels"].add(int(g))  # type: ignore[attr-defined]
                except Exception:
                    pass
    if RX_RETRY.search(text):
        feat["reliability_flags"].add("retry")  # type: ignore[attr-defined]
    if RX_TIMEOUT.search(text):
        feat["reliability_flags"].add("timeout")  # type: ignore[attr-defined]
    if RX_HEADLESS.search(text):
        feat["reliability_flags"].add("headless")  # type: ignore[attr-defined]
    if RX_WAIT_FOR_DEVICE.search(text):
        feat["reliability_flags"].add("wait_for_device")  # type: ignore[attr-defined]
    return feat  # type: ignore[return-value]

def merge_feats(a: Dict, b: Dict) -> Dict:
    out = dict(a)
    out["yaml_loc"] += b["yaml_loc"]
    out["gradle_loc"] += b["gradle_loc"]
    out["matrix_width_hint"] = max(out["matrix_width_hint"], b["matrix_width_hint"])
    out["runs_on"] = set(out["runs_on"]) | set(b["runs_on"])
    out["api_levels"] = set(out["api_levels"]) | set(b["api_levels"])
    out["reliability_flags"] = set(out["reliability_flags"]) | set(b["reliability_flags"])
    return out

# ---------------------------
# Snapshot scan
# ---------------------------
def scan_snapshot_for_styles(repo: Path, treeish: str) -> Tuple[Set[str], Dict]:
    entries = list_tree_entries_cached(str(repo), treeish)
    if not entries:
        return set(), {
            "sources": {"yaml": [], "gradle": [], "scripts_xml": []},
            "features": {
                "yaml_loc": 0, "gradle_loc": 0, "runs_on": [],
                "matrix_width_hint": 0, "api_levels": [], "reliability_flags": [],
            },
        }

    has_community = False
    has_custom = False
    has_gmd_gradle = False
    has_third = False

    seen_yaml: Set[str] = set()
    seen_gradle: Set[str] = set()
    seen_other: Set[str] = set()

    features = {
        "yaml_loc": 0,
        "gradle_loc": 0,
        "runs_on": set(),
        "matrix_width_hint": 0,
        "api_levels": set(),
        "reliability_flags": set(),
    }

    for sha, path in entries:
        if not is_relevant_file(path):
            continue

        text = read_blob_cached(str(repo), sha)
        if not text:
            continue

        # Single content read; use its length for size guards
        if not is_high_priority_ci_path(path) and BLOB_MAX_SIZE and len(text) > BLOB_MAX_SIZE:
            continue
        if len(text) > HARD_MAX_SIZE:
            continue

        f = extract_proxies_from_text(path, text)
        features = merge_feats(features, f)

        if is_gradle(path):
            if not has_gmd_gradle:
                for rx in GMD_GRADLE:
                    if rx.search(text):
                        has_gmd_gradle = True
                        seen_gradle.add("GMD_GRADLE")
                        break
            if not has_community:
                for rx in EMU_COMMUNITY:
                    if rx.search(text):
                        has_community = True
                        seen_gradle.add("Emu_Community")
                        break
            if not has_custom:
                for rx in EMU_CUSTOM:
                    if rx.search(text):
                        has_custom = True
                        seen_gradle.add("Emu_Custom")
                        break

        elif is_yaml(path):
            if not has_community:
                for rx in EMU_COMMUNITY:
                    if rx.search(text):
                        has_community = True
                        seen_yaml.add("Emu_Community")
                        break
            if not has_custom:
                for rx in EMU_CUSTOM:
                    if rx.search(text):
                        has_custom = True
                        seen_yaml.add("Emu_Custom")
                        break
            if not has_third:
                for rx in THIRDPARTY:
                    if rx.search(text):
                        has_third = True
                        seen_yaml.add("ThirdParty")
                        break
            for rx in YAML_VENDOR_EMU_CONTEXT:
                if rx.search(text):
                    seen_yaml.add("EMU_CONTEXT")
                    break

        else:
            if not has_custom and (is_script(path) or is_ci_xml_or_build_xml(path)):
                for rx in EMU_CUSTOM:
                    if rx.search(text):
                        has_custom = True
                        seen_other.add("Emu_Custom")
                        break

    styles: Set[str] = set()
    if has_custom: styles.add("Emu_Custom")
    if has_community: styles.add("Emu_Community")
    if has_third: styles.add("ThirdParty")
    if has_gmd_gradle: styles.add("GMD")

    meta = {
        "sources": {"yaml": sorted(seen_yaml), "gradle": sorted(seen_gradle), "scripts_xml": sorted(seen_other)},
        "features": {
            "yaml_loc": features["yaml_loc"],
            "gradle_loc": features["gradle_loc"],
            "runs_on": sorted(features["runs_on"]),
            "matrix_width_hint": features["matrix_width_hint"],
            "api_levels": sorted(int(x) for x in features["api_levels"]),
            "reliability_flags": sorted(features["reliability_flags"]),
        },
    }
    return styles, meta

# ---------------------------
# Timeline building
# ---------------------------
def empty_result(repo: Path, cutoff_iso: str) -> Dict:
    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "cutoff_heads": [],
        "first_commit_date": None,
        "timeline": [],
        "events": {"Emu_Custom": [], "Emu_Community": [], "GMD": [], "ThirdParty": []},
        "snapshot_as_of_cutoff": [],
        "empty_gaps": [],
    }

def union_snapshot_styles_at_cutoff(repo: Path, cutoff_iso: str, heads: List[str]) -> List[str]:
    styles: Set[str] = set()
    for sha in heads:
        s, _m = scan_snapshot_for_styles(repo, sha)
        styles |= s
    return sorted(styles)

def build_timeline(repo: Path, cutoff_iso: str) -> Dict:
    heads = cutoff_heads(repo, cutoff_iso)
    if not heads:
        return empty_result(repo, cutoff_iso)

    pathspecs_commits = commits_touching_relevant_from_heads(repo, cutoff_iso)
    commit_dates: Dict[str, str] = {}
    for c in pathspecs_commits:
        d = get_commit_date_iso(repo, c)
        if d:
            commit_dates[c] = d
    first_commit_date = commit_dates.get(pathspecs_commits[0]) if pathspecs_commits else None

    timeline: List[Dict] = []
    events: Dict[str, List[Dict]] = {"Emu_Custom": [], "Emu_Community": [], "GMD": [], "ThirdParty": []}
    prev: Set[str] = set()

    empty_gaps: List[Dict] = []
    gap_open: Optional[Dict] = None

    root = earliest_commit(repo)
    if root:
        base_styles, base_meta = scan_snapshot_for_styles(repo, root)
        if base_styles:
            baseline_date = get_commit_date_iso(repo, root)
            timeline.append({"date": baseline_date, "commit": root, "styles": sorted(base_styles), **base_meta})
            for s in sorted(base_styles):
                if s in events:
                    events[s].append({"event": "added", "date": baseline_date, "commit": root})
            prev = set(base_styles)

    for c in pathspecs_commits:
        styles_now, meta_now = scan_snapshot_for_styles(repo, c)
        dt = commit_dates.get(c)

        if SUPPRESS_EMPTY_ROWS and not styles_now:
            if CAPTURE_EMPTY_GAPS and gap_open is None:
                gap_open = {
                    "start_date": dt,
                    "start_commit": c,
                    "prev_state": "+".join(sorted(prev)) if prev else "",
                }
            continue

        if CAPTURE_EMPTY_GAPS and gap_open is not None and styles_now:
            gap_open["end_date"] = dt
            gap_open["end_commit"] = c
            gap_open["next_state"] = "+".join(sorted(styles_now))
            try:
                d1 = datetime.fromisoformat((gap_open["start_date"] or "").replace("Z", "+00:00")).replace(tzinfo=None)  # type: ignore[index]
                d2 = datetime.fromisoformat((dt or "").replace("Z", "+00:00")).replace(tzinfo=None)
                gap_open["duration_days"] = round(max(0.0, (d2 - d1).total_seconds() / 86400.0), 2)
            except Exception:
                gap_open["duration_days"] = None
            gap_open["censored"] = False
            empty_gaps.append(gap_open)
            gap_open = None

        if styles_now != prev:
            if EMIT_NONE_STATE and not styles_now:
                timeline.append({"date": dt, "commit": c, "styles": [], **meta_now})
            elif styles_now:
                timeline.append({"date": dt, "commit": c, "styles": sorted(styles_now), **meta_now})

            added, removed = (styles_now - prev), (prev - styles_now)
            for s in sorted(added):
                if s in events:
                    events[s].append({"event": "added", "date": dt, "commit": c})
            for s in sorted(removed):
                if s in events:
                    events[s].append({"event": "removed", "date": dt, "commit": c})
            prev = styles_now

    snapshot = union_snapshot_styles_at_cutoff(repo, cutoff_iso, heads)

    if CAPTURE_EMPTY_GAPS and gap_open is not None:
        gap_open["end_date"] = cutoff_iso
        gap_open["end_commit"] = heads[0] if heads else None
        gap_open["next_state"] = ""
        try:
            d1 = datetime.fromisoformat((gap_open["start_date"] or "").replace("Z", "+00:00")).replace(tzinfo=None)  # type: ignore[index]
            d2 = datetime.fromisoformat(cutoff_iso.replace("Z", "+00:00")).replace(tzinfo=None)
            gap_open["duration_days"] = round(max(0.0, (d2 - d1).total_seconds() / 86400.0), 2)
        except Exception:
            gap_open["duration_days"] = None
        gap_open["censored"] = True
        empty_gaps.append(gap_open)
        gap_open = None

    qa_issue = None
    if not snapshot:
        qa_issue = "No environment detected at cutoff"

    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "cutoff_heads": heads,
        "first_commit_date": first_commit_date,
        "timeline": timeline,
        "events": events,
        "snapshot_as_of_cutoff": snapshot,
        "empty_gaps": empty_gaps,
        **({"qa_issue": qa_issue} if qa_issue else {}),
    }

# ---------------------------
# Runner + index/summary
# ---------------------------
def output_path_for(repo: Path) -> Path:
    return OUTPUT_MINE / f"{repo.name}.emulator_timeline.json"

def list_repos_once(root: Path) -> List[Path]:
    rs = list_repos(root)
    rs.sort(key=lambda p: p.name.lower())
    return rs

def iso_min(dts: List[str]) -> Optional[str]:
    ds = [d for d in dts if d]
    return min(ds) if ds else None

def derive_summary_fields(rec: Dict) -> Dict:
    name = rec.get("repo_name")
    snapshot = rec.get("snapshot_as_of_cutoff", [])
    events = rec.get("events", {})
    first_events: List[Optional[str]] = []
    for k in ("Emu_Community", "Emu_Custom", "GMD", "ThirdParty"):
        for e in events.get(k, []):
            if e.get("event") == "added":
                first_events.append(e.get("date"))
    first_env_date = iso_min([d for d in first_events if d])
    transitions = 0
    for k in events:
        transitions += sum(1 for e in events[k] if e.get("event") in ("added", "removed"))
    gaps = rec.get("empty_gaps", [])
    gap_days = 0.0
    for g in gaps:
        try:
            gap_days += float(g.get("duration_days") or 0.0)
        except Exception:
            pass
    return {
        "repo_name": name,
        "has_env_at_cutoff": 1 if snapshot else 0,
        "cutoff_states": "+".join(snapshot),
        "first_env_date": first_env_date or "",
        "transitions": transitions,
        "total_gap_days": round(gap_days, 2),
        "qa_issue": rec.get("qa_issue", ""),
    }

def run_miner(cutoff_iso: str = CUTOFF_ISO, max_repos: int = MAX_REPOS) -> None:
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)

    repos = list_repos_once(INPUT_CLONES)
    if max_repos and max_repos > 0:
        repos = repos[:max_repos]

    if not repos:
        print(f"[info] No git repos found under: {INPUT_CLONES}", flush=True)
        return

    print(f"[info] Work root:    {WORK_ROOT}", flush=True)
    print(f"[info] Input repos:  {INPUT_CLONES}", flush=True)
    print(f"[info] Output JSON:  {OUTPUT_MINE}", flush=True)
    print(f"[info] Found repos:  {len(repos)}", flush=True)
    print(f"[info] Cutoff (UTC): {cutoff_iso}", flush=True)
    print(
        f"[info] Options: suppress_empty_rows={SUPPRESS_EMPTY_ROWS}, "
        f"capture_empty_gaps={CAPTURE_EMPTY_GAPS}, emit_none={EMIT_NONE_STATE}, "
        f"blob_max_size={BLOB_MAX_SIZE}",
        flush=True,
    )

    written = skipped = errors = 0
    summaries: List[Dict] = []

    def process(repo: Path):
        rel = repo.relative_to(INPUT_CLONES) if repo != INPUT_CLONES else Path(repo.name)
        print(f"[start] {rel}", flush=True)
        try:
            outp = output_path_for(repo)
            if RESUME_IF_EXISTS and outp.exists():
                return ("skipped", repo, None, None)
            data = build_timeline(repo, cutoff_iso)
            with outp.open("w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            return ("written", repo, data, None)
        except Exception as e:
            return ("error", repo, None, str(e))

    with PoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(process, r): r for r in repos}
        total = len(futs)
        for i, fut in enumerate(as_completed(futs), 1):
            repo = futs[fut]
            rel = repo.relative_to(INPUT_CLONES) if repo != INPUT_CLONES else Path(repo.name)
            status, _repo, data, err = fut.result()
            if status == "written":
                print(f"[{i}/{total}] Wrote: {rel}", flush=True)
                written += 1
                summaries.append(derive_summary_fields(data))  # type: ignore[arg-type]
            elif status == "skipped":
                print(f"[{i}/{total}] Skipped (exists): {rel}", flush=True)
                skipped += 1
                try:
                    with output_path_for(repo).open("r", encoding="utf-8") as f:
                        data2 = json.load(f)
                    summaries.append(derive_summary_fields(data2))
                except Exception:
                    pass
            else:
                print(f"[{i}/{total}] [error] {rel}: {err}", file=sys.stderr, flush=True)
                errors += 1

    # Write summary CSV + JSON index
    idx_json = OUTPUT_MINE / "index_summary.json"
    with idx_json.open("w", encoding="utf-8") as f:
        json.dump({"cutoff": cutoff_iso, "repos": summaries}, f, ensure_ascii=False, indent=2)

    idx_csv = OUTPUT_MINE / "index_summary.csv"
    with idx_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "repo_name",
                "has_env_at_cutoff",
                "cutoff_states",
                "first_env_date",
                "transitions",
                "total_gap_days",
                "qa_issue",
            ],
        )
        w.writeheader()
        for row in summaries:
            w.writerow(row)

    qa_csv = OUTPUT_MINE / "qa_issues.csv"
    with qa_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["repo_name", "qa_issue"])
        w.writeheader()
        for row in summaries:
            if row.get("qa_issue"):
                w.writerow({"repo_name": row["repo_name"], "qa_issue": row["qa_issue"]})

    print(f"[done] JSON -> {OUTPUT_MINE}  (written={written}, skipped={skipped}, errors={errors})", flush=True)
    print(f"[done] Summary: {idx_csv.name}, {idx_json.name}, QA: {qa_csv.name}", flush=True)

# ---------------------------
# ACTUAL EXECUTION (so it *does* start)
# ---------------------------
print("[run] Starting miner…", flush=True)
run_miner(cutoff_iso=CUTOFF_ISO, max_repos=MAX_REPOS)
print("[run] Finished.", flush=True)


[ok] Pre-flight checks passed.
[run] Starting miner…
[info] Work root:    C:\Temp_Instru
[info] Input repos:  C:\Temp_Instru\clonesV9.0
[info] Output JSON:  C:\Temp_Instru\mineV9.0
[info] Found repos:  3
[info] Cutoff (UTC): 2025-08-10 23:59:59 +0000
[info] Options: suppress_empty_rows=True, capture_empty_gaps=True, emit_none=False, blob_max_size=2000000
[start] MetaMask__metamask-mobile
[start] pytorch__executorch
[start] rainbow-me__rainbow
[1/3] Skipped (exists): pytorch__executorch
[2/3] Skipped (exists): MetaMask__metamask-mobile
[3/3] Skipped (exists): rainbow-me__rainbow
[done] JSON -> C:\Temp_Instru\mineV9.0  (written=0, skipped=3, errors=0)
[done] Summary: index_summary.csv, index_summary.json, QA: qa_issues.csv
[run] Finished.
